# MINI-BATCH GRADIENT DESCENT

The main difference between **Batch Gradient Descent (BGD)**, **Stochastic Gradient Descent (SGD)**, and **Mini-Batch Gradient Descent (MBGD)** is **how much training data is used to compute the gradient before updating the model parameters**.

| Feature              | Batch Gradient Descent  | Stochastic Gradient Descent (SGD) | Mini-Batch Gradient Descent  |
| -------------------- | ----------------------- | --------------------------------- | ---------------------------- |
| Data used per update | Entire dataset          | One training example              | Small subset (batch) of data |
| Update frequency     | Once per epoch          | After every example               | After every mini-batch       |
| Speed                | Slow for large datasets | Very fast updates                 | Balanced                     |
| Memory usage         | High                    | Low                               | Moderate                     |
| Stability            | Very stable             | Noisy updates                     | Less noisy than SGD          |
| Convergence          | Smooth but can be slow  | Fast initially but fluctuates     | Fast and stable              |

### 1. Batch Gradient Descent (BGD)

* Uses **all training samples** to calculate the gradient.
* Updates the model **once after processing the entire dataset**.

**Example:**
Suppose you have **10,000 training examples**.

* Compute loss on all 10,000 examples.
* Calculate gradient.
* Update weights once.

```
Epoch 1:
10000 samples → Compute gradient → Update weights
```

**Pros**

* Stable and smooth convergence.
* Accurate gradient estimate.

**Cons**

* Slow for large datasets.
* Requires lots of memory.

---

### 2. Stochastic Gradient Descent (SGD)

* Uses **only one training example** at a time.
* Updates weights **after every sample**.

For 10,000 examples:

```
Sample 1 → Update
Sample 2 → Update
Sample 3 → Update
...
Sample 10000 → Update
```

So there are **10,000 updates in one epoch**.

**Pros**

* Very fast updates.
* Can escape local minima due to noisy gradients.
* Suitable for online learning.

**Cons**

* Loss fluctuates a lot.
* Takes a zigzag path toward the minimum.

---

### 3. Mini-Batch Gradient Descent (Most Common)

* Uses a **small batch** of samples (e.g., 32, 64, 128, 256).
* Updates weights after each mini-batch.

Suppose:

* Dataset = **10,000 samples**
* Batch size = **100**

Then:

```
Batch 1 (100 samples) → Update
Batch 2 (100 samples) → Update
...
Batch 100 → Update
```

So there are **100 updates per epoch**.

**Pros**

* Faster than batch gradient descent.
* More stable than SGD.
* Efficient on GPUs.
* Most widely used in deep learning.

**Cons**

* Still has some noise compared with full-batch updates.

---

## Visual Comparison

```
Batch GD
Entire Dataset
[1 2 3 ... 10000]
          ↓
     One Update

--------------------------

Stochastic GD
1 → Update
2 → Update
3 → Update
...
10000 → Update

--------------------------

Mini-Batch GD (batch size = 100)

1-100     → Update
101-200   → Update
201-300   → Update
...
9901-10000 → Update
```

### Summary

* **Batch Gradient Descent:** Uses **100% of the data** for each update. Accurate but slow.
* **Stochastic Gradient Descent (SGD):** Uses **1 sample** per update. Fast but noisy.
* **Mini-Batch Gradient Descent:** Uses a **small batch** (commonly 32–256 samples) per update. It offers a good balance of speed, stability, and efficiency, which is why it is the standard choice for training most modern machine learning and deep learning models.


In [274]:
from sklearn.datasets import load_diabetes
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [275]:
X, y = load_diabetes(return_X_y=True)

In [276]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [277]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=2)

In [278]:
X_train.shape

(353, 10)

In [279]:
import random
class Mini_BatchGDReg:

    def __init__(self,batch_size, learning_rate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X_train, y_train):
        # Initialize parameters
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])

        for i in range(self.epochs):
            for j in range(int(X_train.shape[0]/self.batch_size)):
                idx = random.sample(range(X_train.shape[0]), self.batch_size)

                # Prediction
                y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_
                intercept_der = -2 * np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)
                coef_der = -2 * np.dot((y_train[idx] - y_hat),X_train[idx])

                # Update parameters
                self.coef_ -= self.lr * coef_der

        print("Intercept:", self.intercept_)
        print("Coefficients:", self.coef_)

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [280]:
mbdgr = Mini_BatchGDReg(batch_size=int(X_train.shape[0]/10), learning_rate=0.01, epochs=50)

In [281]:
mbdgr.fit(X_train, y_train)

Intercept: 151.89428402064047
Coefficients: [  54.29501457  -68.82272863  345.24169284  240.0503413    17.94151342
  -30.35731079 -166.60753277  125.48212207  318.31959376  127.46616558]


In [282]:
y_pred = mbdgr.predict(X_test)

In [283]:
r2_score(y_test,y_pred)

0.4331075453554044

### Now using sklearn Gradient descent libs 

In [286]:
from sklearn.linear_model import SGDRegressor

In [287]:
sgd = SGDRegressor(learning_rate='constant', eta0=0.2)

In [289]:
batch_size = 35
for i in range(100):
    idx = random.sample(range(X_train.shape[0]), batch_size)
    sgd.partial_fit(X_train[idx], y_train[idx])

In [290]:
sgd.coef_

array([  28.87646399, -102.67040497,  445.4439394 ,  283.79442899,
        -26.12318769, -107.99301946, -197.15289814,  111.43887119,
        434.22846885,   99.73567246])

In [291]:
sgd.intercept_

array([155.73802072])

In [292]:
y_pred2 = sgd.predict(X_test)

In [293]:
r2_score(y_test,y_pred2)

0.4451202094201385